In [1]:
# Module imports
import pandas as pd
import numpy as np
import mne
import os
import scipy as sc
from scipy import signal
from scipy.signal import butter, lfilter, filtfilt
import matplotlib.pyplot as plt
import plotly.io as pio
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
pio.renderers.default='browser'

from physiological_signal_functions import load_biopac_data, process_biopac_data, extract_adalearn_triggers_biopac, extract_task_session_emg_data

In [2]:
# Define participant, condition and task session characteristics
# Participants parameters
participant = 1
participant_str = str(participant).zfill(2)
list_subject_selected = [participant]

# Condition 1 (c1) characteristics
condition_1 = str(1).zfill(2)
c1_run_ranges = [
    {'run_start': '01', 'run_end': '04'},
    {'run_start': '05', 'run_end': '08'},
    {'run_start': '09', 'run_end': '12'},
    {'run_start': '13', 'run_end': '16'},
    {'run_start': '17', 'run_end': '20'}
]

# Condition 2 (c2) characteristics
condition_2 = str(2).zfill(2)
c2_run_ranges = [
    {'run_start': '01', 'run_end': '04'},
    {'run_start': '05', 'run_end': '08'},
    {'run_start': '09', 'run_end': '12'},
    {'run_start': '13', 'run_end': '16'}
]

In [3]:
# Theoretical stimulation parameters
stimfreq     = 30
ONduration   = 30
OFFduration = 300
minnumpulse  = (ONduration+2) / (1/stimfreq) # Two stimulations added to data files to signal start and end of task
print('The stimulation parameters should be:')
print('Stimulation frequency :', stimfreq, 'Hz')
print('ON duration:', ONduration, 's')
print('OFF duration:', OFFduration, 's')
print('Minimum number of pulses per stimulation', minnumpulse)

The stimulation parameters should be:
Stimulation frequency : 30 Hz
ON duration: 30 s
OFF duration: 300 s
Minimum number of pulses per stimulation 960.0


In [4]:
# Loading biopac data using function in physiological_signal_functions
biopac_data_c1, stim_channel, trig_channel = load_biopac_data(participant_str, condition_1, c1_run_ranges)
biopac_data_c2, stim_channel, trig_channel = load_biopac_data(participant_str, condition_2, c2_run_ranges)

Current working directory: /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-01
Loading file: sub-01_ses-01_task-AdalearnVNS_run-01to04_physio.edf
Extracting EDF parameters from /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-01/sub-01_ses-01_task-AdalearnVNS_run-01to04_physio.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


/home/aaron/Documents/Python /Ada_Learn/Ada_Learn Analysis/Analysis_scripts_functions/physiological_signal_functions.py:53: RuntimeWarning:

Channel names are not unique, found duplicates for: {'Digital (STP Inp'}. Applying running numbers for duplicates.



Reading 0 ... 1915999  =      0.000 ...   958.000 secs...
Channel names in sub-01_ses-01_task-AdalearnVNS_run-01to04_physio.edf: ['ECG - ECG100C', 'STIM - EMG100C', 'Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Stim channels found: ['STIM - EMG100C']
Trig channels found: ['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Loading file: sub-01_ses-01_task-AdalearnVNS_run-05to08_physio.edf
Extracting EDF parameters from /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-01/sub-01_ses-01_task-AdalearnVNS_run-05to08_physio.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1459999  =      0.000 ...   730.000 secs...


/home/aaron/Documents/Python /Ada_Learn/Ada_Learn Analysis/Analysis_scripts_functions/physiological_signal_functions.py:53: RuntimeWarning:

Channel names are not unique, found duplicates for: {'Digital (STP Inp'}. Applying running numbers for duplicates.



Channel names in sub-01_ses-01_task-AdalearnVNS_run-05to08_physio.edf: ['ECG - ECG100C', 'STIM - EMG100C', 'Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Stim channels found: ['STIM - EMG100C']
Trig channels found: ['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Loading file: sub-01_ses-01_task-AdalearnVNS_run-09to12_physio.edf
Extracting EDF parameters from /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-01/sub-01_ses-01_task-AdalearnVNS_run-09to12_physio.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1611999  =      0.000 ...   806.000 secs...


...

Channel names in sub-01_ses-02_task-AdalearnVNS_run-05to08_physio.edf: ['ECG - ECG100C', 'STIM - EMG100C', 'Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Stim channels found: ['STIM - EMG100C']
Trig channels found: ['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Loading file: sub-01_ses-02_task-AdalearnVNS_run-09to12_physio.edf
Extracting EDF parameters from /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-02/sub-01_ses-02_task-AdalearnVNS_run-09to12_physio.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1513999  =      0.000 ...   757.000 secs...


/home/aaron/Documents/Python /Ada_Learn/Ada_Learn Analysis/Analysis_scripts_functions/physiological_signal_functions.py:53: RuntimeWarning:

Channel names are not unique, found duplicates for: {'Digital (STP Inp'}. Applying running numbers for duplicates.



Channel names in sub-01_ses-02_task-AdalearnVNS_run-09to12_physio.edf: ['ECG - ECG100C', 'STIM - EMG100C', 'Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Stim channels found: ['STIM - EMG100C']
Trig channels found: ['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Loading file: sub-01_ses-02_task-AdalearnVNS_run-13to16_physio.edf
Extracting EDF parameters from /home/aaron/Documents/VNS Ada_Learn/data_VNS/sub-01/ses-02/sub-01_ses-02_task-AdalearnVNS_run-13to16_physio.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 1893999  =      0.000 ...   947.000 secs...


/home/aaron/Documents/Python /Ada_Learn/Ada_Learn Analysis/Analysis_scripts_functions/physiological_signal_functions.py:53: RuntimeWarning:

Channel names are not unique, found duplicates for: {'Digital (STP Inp'}. Applying running numbers for duplicates.



Channel names in sub-01_ses-02_task-AdalearnVNS_run-13to16_physio.edf: ['ECG - ECG100C', 'STIM - EMG100C', 'Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Stim channels found: ['STIM - EMG100C']
Trig channels found: ['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Final stim channel: STIM - EMG100C
['Digital (STP Inp-0', 'Digital (STP Inp-1', 'Digital (STP Inp-2', 'Digital (STP Inp-3', 'Digital (STP Inp-4', 'Digital (STP Inp-5', 'Digital (STP Inp-6']
Final trig channel: Digital (STP Inp-2


In [5]:
# Processing EMG data and storing it in a dataframe using function in physiological_signal_functions
processed_biopac_c1 = process_biopac_data(biopac_data_c1, stim_channel)
processed_biopac_c2 = process_biopac_data(biopac_data_c2,stim_channel)

In [6]:
# Extracting adalearn outcome presentation triggers within bipoac data using function in physiological_signal_functions 
outcome_triggers_c1, c1_trig_summary = extract_adalearn_triggers_biopac(biopac_data_c1, trig_channel, plot=False)
outcome_triggers_c2, c2_trig_summary = extract_adalearn_triggers_biopac(biopac_data_c2, trig_channel, plot=False)

Number of valid triggers found: 1540
Number of triggers in each task session:
task_session
1     77
2     77
3     77
4     77
5     77
6     77
7     77
8     77
9     77
10    77
11    77
12    77
13    77
14    77
15    77
16    77
17    77
18    77
19    77
20    77
Name: count, dtype: int64
Number of valid triggers found: 1232
Number of triggers in each task session:
task_session
1     77
2     77
3     77
4     77
5     77
6     77
7     77
8     77
9     77
10    77
11    77
12    77
13    77
14    77
15    77
16    77
Name: count, dtype: int64


In [7]:
# Reducing EMG data to individual dataframes related to the duration of a task session and storing this within a dictionary using a function within physiological_signal_functions 
c1_task_sess_dict = extract_task_session_emg_data(processed_biopac_c1, c1_trig_summary, plot=True)
c2_task_sess_dict = extract_task_session_emg_data(processed_biopac_c2, c2_trig_summary, plot=True)

# Checking the length of the dictionary for condition 1
print(f"In condition {condition_1} the number of task sessions is: {len(c1_task_sess_dict)}")
# Checking the shape of each dataframe in the dictionary
for key, df in c1_task_sess_dict.items():
    print(f"Task Session {key} shape: {df.shape}")

# Checking the length of the dictionary for condition 2
print(f"In condition {condition_2} the number of task sessions is: {len(c2_task_sess_dict)}")
# Checking the shape of each dataframe in the dictionary
for key, df in c2_task_sess_dict.items():
    print(f"Task Session {key} shape: {df.shape}")

In condition 01 the number of task sessions is: 20
Task Session 1 shape: (227553, 5)
Task Session 2 shape: (227629, 5)
Task Session 3 shape: (227572, 5)
Task Session 4 shape: (227611, 5)
Task Session 5 shape: (227368, 5)
Task Session 6 shape: (227563, 5)
Task Session 7 shape: (227607, 5)
Task Session 8 shape: (227398, 5)
Task Session 9 shape: (227475, 5)
Task Session 10 shape: (227454, 5)
Task Session 11 shape: (227329, 5)
Task Session 12 shape: (227421, 5)
Task Session 13 shape: (227503, 5)
Task Session 14 shape: (227257, 5)
Task Session 15 shape: (227434, 5)
Task Session 16 shape: (227453, 5)
Task Session 17 shape: (227381, 5)
Task Session 18 shape: (227256, 5)
Task Session 19 shape: (227523, 5)
Task Session 20 shape: (227891, 5)
In condition 02 the number of task sessions is: 16
Task Session 1 shape: (229045, 5)
Task Session 2 shape: (229130, 5)
Task Session 3 shape: (229060, 5)
Task Session 4 shape: (229088, 5)
Task Session 5 shape: (229047, 5)
Task Session 6 shape: (229066, 5)
Tas

In [6]:
num_files = len(processed_biopac_c1['data_file'].unique()) - 4

# Create subplot titles
subplot_titles = []
for i in range(num_files):
    subplot_titles.extend([f"File {i + 1} - Raw EMG", f"File {i + 1} - Rectified EMG", f"File {i + 1} - Enveloped EMG"])

# Plotting the data across individual data files
fig = make_subplots(rows=3, cols=num_files, subplot_titles=subplot_titles,
                    shared_xaxes=True, vertical_spacing=0.1,
                    row_titles=["Raw EMG", "Rectified EMG", "Enveloped EMG"])

for i in range(num_files):
    file_data = processed_biopac_c1[processed_biopac_c1['data_file'] == (i + 1)]
    
    # Check the size of the data
    print(f"Data file {i + 1} size: {file_data.shape}")

    # Add traces for each subplot
    fig.add_trace(go.Scatter(x=file_data['time'], y=file_data['emg'], name=f'Raw EMG {i+1}'), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=file_data['time'], y=file_data['rectified_emg'], name=f'Rectified EMG {i+1}'), row=2, col=i+1)
    fig.add_trace(go.Scatter(x=file_data['time'], y=file_data['lowpass_filtered'], name=f'Enveloped EMG {i+1}'), row=3, col=i+1)

fig.update_layout(height=900, width=1200, title_text="EMG Data Per Data File", showlegend=False)

fig.show()


Data file 1 size: (1916000, 5)


In [9]:
num_files = len(outcome_triggers_c1['data_file'].unique()) - 4


for i in range(num_files):
    file_data = outcome_triggers_c1[outcome_triggers_c1['data_file'] == (i + 1)]
    
    # Check the size of the data
    print(f"Data file {i + 1} size: {file_data.shape}")

    # Add traces for each subplot
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=file_data['time'], y=file_data['triggers'], name=f'Triggers {i+1}'))

fig.update_layout(height=900, width=1200, title_text="Triggers Per Data File", showlegend=False)

fig.show()

Data file 1 size: (1916000, 3)
